# 02 – PINN training

Data and training of the PINN (inputs: t, R0, Delta; outputs: S, E, I, R, D, Dobs for the 16 age classes).

- **Training set**: deterministic trajectories on a 33 x 33 grid of (R0, Delta), weekly time grid, one infected per age class at t = 0.
- **Collocation set**: midpoints of the parameter grid, daily time grid. The ODE residual is evaluated on these points.

Two modes, set by `REGENERATE_DATA` below:

- `False` (default): load the files in `DATA_DIR` (`data/training/`). If they are missing (the training CSV, 118 MB, is not stored in the repository), they are generated.
- `True`: generate both sets again and save them in `DATA_DIR`.

Then the network is trained. The weights, the losses of every epoch and the run time are saved in `OUT_DIR`. To use the weights without training, copy them to `precomputed/pretrained_pinn/`.

In [2]:
import os
import numpy as np
import torch
from torch import nn

from pinnvsmech.constants import ifr_hetero, N_age, xi, epsilon
from pinnvsmech.data.load_matrices import load_contact_matrix
from pinnvsmech.data.synthetic import generate_deterministic_data, save_deterministic_data
from pinnvsmech.data.pinn_training_data import (
    parameter_grid, generate_collocation_set, save_collocation_set, load_collocation_set,
    TRAIN_CSV, TRAIN_PAIRS_PICKLE, COLLOCATION_PICKLE,
)
from pinnvsmech.pinn.NN_free_params import NN
from pinnvsmech.pinn.approximation import load_train_data
from pinnvsmech.pinn.training import train_pinn
from pinnvsmech.paths import TRAINING_DIR, OUTPUTS_DIR

torch.set_default_dtype(torch.float64)

In [ ]:
# ---- Settings -------------------------------------------------------------
REGENERATE_DATA = False                # True: generate training and collocation sets again; False: load them
DATA_DIR = TRAINING_DIR                # where to read/write. Use OUTPUTS_DIR / "training_data" to avoid overwriting data/training
OUT_DIR = OUTPUTS_DIR / "pinn"         # weights, losses and run time of the training

# Only used if REGENERATE_DATA = True
COUNTRY = "United Kingdom of Great Britain"   # contact matrix (Prem et al.)
R0_RANGE = (1.3, 3.0)
DELTA_RANGE = (7, 35)                         # mean delay, days
N_P_1D = 33                                   # grid size per parameter (33 x 33 = 1089 scenarios)
DOMAIN = (0, 365)                             # days

# Training
N_EPOCHS = 4000
LR = 1e-3
WEIGHT_DECAY = 0
BATCH_SIZE = 100
TRAIN_SIZE = 0.9                              # the rest is used for validation
ALPHA = {'ode': 100, 'mse': 1}               # loss weights
HIDDEN = [100, 100, 100, 100, 100]            # hidden layers

k_age = len(N_age)
RUN_NAME = f"pinn_I0ones_lossODE{ALPHA['ode']}_epochs{N_EPOCHS}"

## 1. Data

In [3]:
C = load_contact_matrix(COUNTRY)     # needed by the loss, also when the data are loaded

_missing = not (os.path.exists(os.path.join(DATA_DIR, TRAIN_CSV)) and os.path.exists(os.path.join(DATA_DIR, COLLOCATION_PICKLE)))
if _missing and not REGENERATE_DATA:
    print("Data files not found in", DATA_DIR, "-> generating them")

if REGENERATE_DATA or _missing:
    # Training set
    p_train = parameter_grid(R0_RANGE, DELTA_RANGE, N_P_1D)
    x_train, y_train, idx_train = generate_deterministic_data(
        p_train, C, N_age, ifr_hetero, epsilon, xi, domain=DOMAIN, step=7, ic_mode="standard")
    save_deterministic_data(x_train, y_train, idx_train, p_train, folder=DATA_DIR,
                            csv_name=TRAIN_CSV, pairs_name=TRAIN_PAIRS_PICKLE)

    # Collocation set
    x_coll_np = generate_collocation_set(p_train, domain=DOMAIN)
    save_collocation_set(x_coll_np, folder=DATA_DIR, name=COLLOCATION_PICKLE)
else:
    print("Skipped: loading existing data from", DATA_DIR)

/home/mrama/venvs/pinns/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Data files not found in /home/mrama/SINNs/PINNs_epi/Repo_GitHub/PINNvsMechanistic-models-for-epidemic-modeling-and-forecasting/data/training -> generating them


Generating training data:   0%|          | 0/1089 [00:00<?, ?it/s]

In [4]:
x_train_np, y_train_np, idx_train_np = load_train_data(os.path.join(DATA_DIR, TRAIN_CSV), k_age=k_age)
x_coll_np = load_collocation_set(DATA_DIR, COLLOCATION_PICKLE)

x_train = torch.tensor(x_train_np, dtype=torch.float64)
y_train = torch.tensor(y_train_np, dtype=torch.float64)
x_coll_full = torch.tensor(x_coll_np, dtype=torch.float64)
print("x_coll:", tuple(x_coll_full.shape))

x_train   shape: (57717, 3)
y_train   shape: (57717, 96)
idx_train shape: (57717, 1)
x_coll: (374784, 3)


## 2. Network

In [5]:
dimensions = [3] + HIDDEN + [6 * k_age]
model = NN(activation=nn.Tanh, dimensions=dimensions)
print(dimensions)

[3, 100, 100, 100, 100, 100, 96]


## 3. Training

In [ ]:
history, runtime_log = train_pinn(
    model, x_train, y_train, x_coll_full, C, N_age, epsilon, xi, ifr_hetero,
    alpha=ALPHA, n_epochs=N_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE, train_size=TRAIN_SIZE,
    model_path=OUT_DIR / "models" / f"{RUN_NAME}.pth",
    log_csv=OUT_DIR / "logs" / f"Losses_{RUN_NAME}.csv",
    runtime_json=OUT_DIR / "logs" / f"runtime_log_{RUN_NAME}.json",
)

Training Completed:   0%, Average Train Loss:  8.10e-02, Average Val Loss:  3.68e-02
